# MLP Sanity Checks — What can the Lagrangian MLP actually learn?

This notebook trains the MLP **inline** (no external script) and runs a series of sanity checks to understand what the corrector can and cannot capture, before committing to the on-the-fly fine-tuning pipeline.

| Section | Question |
|---------|----------|
| 0 | Config & data |
| 1 | Target signal anatomy — what is ΔF made of? |
| 2 | Baseline A: predict-zero and predict-mean |
| 3 | Baseline B: predict Lagrangian displacement (pos − q₀) |
| 4 | Feature–ΔF correlation — what does the MLP see? |
| 5 | **MLP training in-notebook** — live convergence curve |
| 6 | What did the MLP learn? Scatter + spatial maps |
| 7 | Context window scan — train R vs test R vs n_shell |
| 8 | CNN + MLP comparison (if CNN checkpoint available) |
| 9 | Summary table — all baselines vs MLP vs CNN+MLP |

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pickle, time
from pathlib import Path
from types import SimpleNamespace
from functools import partial

import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr

import yaml

REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT / "pm2nbody"))

from train_lag_force import (
    compute_force_pair,
    snapshot_features,
    make_train_step,
    compute_sample_weights,
)
from train_subregion_forceres import load_single_snapshot, make_patch_split
from train_lag_massres import (
    _load_cnn_massres_checkpoint,
    compute_cnn_massres_correction,
)
from jaxpm.lagrangian import (
    get_axis_neighbor_indices,
    get_shell_neighbor_indices,
    make_lagrangian_corrector,
)

print("imports OK  |  JAX:", jax.devices())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIG — edit here
# ═══════════════════════════════════════════════════════════════════
CONFIG_PATH = REPO_ROOT / "configs/subregion_forceres.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

data_cfg  = SimpleNamespace(**cfg["data"])
model_cfg = SimpleNamespace(**cfg["model"])
train_cfg = SimpleNamespace(**cfg["training"])

DATA_DIR      = Path(data_cfg.data_dir)
N_PART        = int(data_cfg.n_part)
MESH_LR       = int(data_cfg.mesh_lr)
MESH_HR       = int(data_cfg.mesh_hr)
BOX_SIZE      = float(data_cfg.box_size)
SIM_TRAIN     = int(data_cfg.sim_id_train)
SIM_VAL       = int(getattr(data_cfg, "sim_id_val", 1))
SNAP_TRAIN    = int(data_cfg.snap_train)
SNAPS_VAL     = list(data_cfg.snaps_val)
TRAIN_PATCH_N = int(train_cfg.train_patch_n)
PATCH_SEED    = int(getattr(train_cfg, "patch_seed", 42))
USE_STRAIN    = bool(model_cfg.use_strain)
USE_INVS      = bool(model_cfg.use_invariants)
USE_VEL       = bool(model_cfg.use_velocity)
CNN_CKPT_PATH = getattr(model_cfg, "cnn_checkpoint", None)
SCALE_TO_LR   = float(MESH_LR) / float(N_PART)

# ── MLP hyperparams (edit to test different sizes) ───────────────────────────
HIDDEN_DIM = int(model_cfg.hidden_dim)    # start: 64
N_LAYERS   = int(model_cfg.n_layers)      # start: 3
N_SHELL    = int(getattr(model_cfg, "n_shell", 0))  # 0=axis only, 1=+18, 2=+56
N_STEPS    = 2000
LR         = 3e-4
LOG_EVERY  = 50

# ── Split ────────────────────────────────────────────────────────────────────
neighbor_idx = get_axis_neighbor_indices(N_PART)
train_idx, test_idx, _ = make_patch_split(N_PART, TRAIN_PATCH_N, seed=PATCH_SEED)
train_frac = (TRAIN_PATCH_N / N_PART) ** 3

print(f"n_part={N_PART}  mesh_lr={MESH_LR}  mesh_hr={MESH_HR}")
print(f"train_patch_n={TRAIN_PATCH_N}  ({train_frac:.1%} of sim)")
print(f"MLP: hidden={HIDDEN_DIM}  layers={N_LAYERS}  n_shell={N_SHELL}")
print(f"CNN: {'enabled — ' + str(CNN_CKPT_PATH) if CNN_CKPT_PATH else 'disabled'}") 

In [ ]:
# ── Load snapshot ─────────────────────────────────────────────────────────────
pos, vel, a = load_single_snapshot(DATA_DIR, SIM_TRAIN, SNAP_TRAIN, N_PART, BOX_SIZE)
pos_lr = pos * SCALE_TO_LR

# ── Force pair (unit-corrected) ───────────────────────────────────────────────
_fp = jax.jit(partial(compute_force_pair, mesh_lr=MESH_LR, mesh_hr=MESH_HR))
f_coarse, f_fine, delta_f = _fp(pos_lr)

f_coarse_np = np.asarray(jax.device_get(f_coarse))
f_fine_np   = np.asarray(jax.device_get(f_fine))
delta_f_np  = np.asarray(jax.device_get(delta_f))
pos_np      = np.asarray(jax.device_get(pos))

# ── Lagrangian features ───────────────────────────────────────────────────────
ext_idx, ext_off, shell_sl = None, None, ()
if N_SHELL > 0:
    ext_idx, ext_off, shell_sl = get_shell_neighbor_indices(N_PART, N_SHELL)

feats_all, det_D_all = snapshot_features(
    pos, neighbor_idx, N_PART, USE_STRAIN, USE_INVS,
    ext_idx, ext_off, "mean_var", shell_sl,
)
feats_np = np.asarray(jax.device_get(feats_all))
det_D_np = np.asarray(det_D_all)

# ── Lagrangian initial lattice position q₀ ────────────────────────────────────
# Particles start on a regular grid; flat index m = ix*N²+iy*N+iz
q0_idx   = np.arange(N_PART**3)
q0_np    = np.stack(np.unravel_index(q0_idx, (N_PART,)*3), axis=-1).astype(np.float32)
displace_np = pos_np - q0_np   # Eulerian pos − initial lattice (n_part units)

df_mag = np.sqrt(np.sum(delta_f_np**2, axis=-1))
print(f"a={a:.4f}  feat_dim={feats_np.shape[1]}  SC={np.mean(det_D_np<0):.2%}")
print(f"|ΔF| mean={df_mag.mean():.4e}  max={df_mag.max():.4e}")
print(f"|displacement| mean={np.sqrt(np.sum(displace_np**2,axis=-1)).mean():.4e}")

## Section 1 — Target signal anatomy

Before training anything, understand what ΔF looks like spatially and statistically.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1: ΔF distributions (train vs test)
bins = np.linspace(*np.percentile(delta_f_np[:,0], [0.5, 99.5]), 100)
for ci, comp in enumerate(["x", "y", "z"]):
    ax = axes[0, ci]
    ax.hist(delta_f_np[train_idx, ci], bins=bins, alpha=0.6,
            density=True, color="tomato",    label="train")
    ax.hist(delta_f_np[test_idx,  ci], bins=bins, alpha=0.6,
            density=True, color="steelblue", label="test")
    # Mark means
    ax.axvline(delta_f_np[train_idx, ci].mean(), color="tomato",    ls="--", lw=1.5,
               label=f"train mean={delta_f_np[train_idx, ci].mean():.3f}")
    ax.axvline(delta_f_np[test_idx,  ci].mean(), color="steelblue", ls="--", lw=1.5,
               label=f"test  mean={delta_f_np[test_idx,  ci].mean():.3f}")
    ax.set_title(f"ΔF_{comp}"); ax.legend(fontsize=7)

# Row 2: |ΔF| vs local density proxy (strain magnitude)
strain_mag = np.sqrt(np.sum(feats_np[:, :9]**2, axis=-1)) if feats_np.shape[1] >= 9 else np.ones(len(feats_np))
ss = np.random.default_rng(0).choice(len(df_mag), min(40_000, len(df_mag)), replace=False)
for ci, (xdata, xlabel) in enumerate([
    (strain_mag, "strain |E|"),
    (np.abs(det_D_np), "|det D|"),
    (np.sqrt(np.sum(displace_np**2, axis=-1)), "|displacement|"),
]):
    ax = axes[1, ci]
    h  = ax.hexbin(xdata[ss], df_mag[ss], gridsize=55, cmap="plasma",
                   norm=LogNorm(), mincnt=1)
    plt.colorbar(h, ax=ax)
    ax.set_xlabel(xlabel); ax.set_ylabel("|ΔF|")
    r, _ = pearsonr(xdata[ss], df_mag[ss])
    ax.set_title(f"|ΔF| vs {xlabel}  R={r:.3f}")

plt.suptitle(f"ΔF signal anatomy  a={a:.3f}", fontsize=12)
plt.tight_layout(); plt.show()

print("\n── Train vs Test ΔF statistics ─────────────────────────────")
for region, idx in [("train", train_idx), ("test", test_idx)]:
    df_r = delta_f_np[idx]
    print(f"{region:>6}:  mean=({df_r[:,0].mean():.3f}, {df_r[:,1].mean():.3f}, {df_r[:,2].mean():.3f})  "
          f"|ΔF| mean={np.sqrt(np.sum(df_r**2,axis=-1)).mean():.4e}")

## Section 2 — Baseline A: predict-zero and predict-mean

**Predict-zero**: always output ΔF=0. Baseline for MSE improvement.  
**Predict-mean (train)**: output the mean ΔF of the training region to all test particles.  
If predict-mean beats predict-zero on the test region, the mean is spatially correlated — the patch and the rest share the same large-scale mode.

In [ ]:
def r_mean_3d(pred, tgt):
    """Mean Pearson R across 3 force components."""
    return float(np.mean([pearsonr(pred[:,c], tgt[:,c])[0] for c in range(3)]))

def mse_improv(pred, tgt_f_fine, f_coarse_np):
    """MSE improvement of (f_coarse + pred) vs f_fine, relative to f_coarse baseline."""
    base = np.mean((f_coarse_np - tgt_f_fine)**2)
    corr = np.mean((f_coarse_np + pred - tgt_f_fine)**2)
    return 1 - corr / base

# ── Predict-zero ──────────────────────────────────────────────────────────────
pred_zero = np.zeros_like(delta_f_np)

# ── Predict-mean (train) ──────────────────────────────────────────────────────
train_mean = delta_f_np[train_idx].mean(axis=0)   # [3]  vectorial mean
pred_mean  = np.tile(train_mean, (N_PART**3, 1))   # broadcast to all particles

# ── Predict-mean-per-component-sign (captures bulk flow direction) ─────────────
train_median = np.median(delta_f_np[train_idx], axis=0)
pred_median  = np.tile(train_median, (N_PART**3, 1))

print(f"Training region mean ΔF: ({train_mean[0]:.4f}, {train_mean[1]:.4f}, {train_mean[2]:.4f})")
print(f"Test     region mean ΔF: ({delta_f_np[test_idx,0].mean():.4f}, "
      f"{delta_f_np[test_idx,1].mean():.4f}, {delta_f_np[test_idx,2].mean():.4f})")
print()

results = []
for name, pred in [("predict-zero", pred_zero), ("predict-train-mean", pred_mean),
                   ("predict-train-median", pred_median)]:
    for region, idx in [("train", train_idx), ("test", test_idx)]:
        r  = r_mean_3d(pred[idx], delta_f_np[idx])
        mi = mse_improv(pred[idx], f_fine_np[idx], f_coarse_np[idx])
        results.append({"baseline": name, "region": region, "R": r, "MSE_improv%": mi*100})

print(f"{'Baseline':<24} {'Region':<8} {'R':>8} {'MSE_improv%':>12}")
print("─" * 56)
for row in results:
    print(f"{row['baseline']:<24} {row['region']:<8} {row['R']:>8.4f} {row['MSE_improv%']:>11.1f}%")

## Section 3 — Baseline B: predict Lagrangian displacement

The Lagrangian displacement **d = x − q₀** is already embedded in the features (deformation tensor). If a simple linear fit from features → displacement beats the MLP on forces, the features carry strong spatial information but the MLP is bottlenecked by the force target complexity.

We also check: **linear regression from features → ΔF** (upper bound on what the MLP can learn with linear weights).

In [ ]:
from numpy.linalg import lstsq

def linear_fit_r(feats, target, train_idx, test_idx, label):
    """Fit linear map feats[train] → target[train], evaluate R on test."""
    X_tr = feats[train_idx]       # [N_tr, D]
    y_tr = target[train_idx]      # [N_tr, 3]
    X_te = feats[test_idx]
    y_te = target[test_idx]

    # Add bias
    Xb_tr = np.hstack([X_tr, np.ones((len(X_tr), 1))])
    Xb_te = np.hstack([X_te, np.ones((len(X_te), 1))])

    W, _, _, _ = lstsq(Xb_tr, y_tr, rcond=None)   # [D+1, 3]
    pred_tr = Xb_tr @ W
    pred_te = Xb_te @ W

    r_tr = r_mean_3d(pred_tr, y_tr)
    r_te = r_mean_3d(pred_te, y_te)
    print(f"{label:<36} train R={r_tr:.4f}  test R={r_te:.4f}  gap={r_tr-r_te:+.4f}")
    return pred_tr, pred_te, W

print("── Linear regression (features → target) ─────────────────────────────")
print(f"{'Target':<36} {'train R':>8}  {'test R':>8}  {'gap':>8}")
print("─" * 70)

# A: features → ΔF (what the MLP approximates with a linear model)
_, pred_lm_df_te, _ = linear_fit_r(feats_np, delta_f_np, train_idx, test_idx,
                                    "linear: feats → ΔF")

# B: features → displacement (easier target — smoother, long-range)
_, pred_lm_disp_te, _ = linear_fit_r(feats_np, displace_np, train_idx, test_idx,
                                      "linear: feats → displacement")

# C: features → |ΔF| (scalar, what drives per-particle weight)
df_mag_col = df_mag[:, None]
_, pred_lm_mag_te, _ = linear_fit_r(feats_np, df_mag_col, train_idx, test_idx,
                                     "linear: feats → |ΔF|")

print()
print("Interpretation:")
print("  linear feats→ΔF R  ≈ upper bound on MLP R if features are sufficient")
print("  linear feats→disp  = how well features encode the large-scale flow")

## Section 4 — Feature–ΔF correlation

Pearson R of each feature vs each ΔF component.  
Features with high |R| in both train and test → MLP can generalise on those.  
Features strong in train but weak in test → spatial overfitting risk.

In [ ]:
feat_dim = feats_np.shape[1]
feat_names = [f"E_{ab}" for ab in ["xx","xy","xz","yx","yy","yz","zx","zy","zz"]]
feat_names += [f"f{i}" for i in range(feat_dim - len(feat_names))]
feat_names = feat_names[:feat_dim]

r_tr = np.array([pearsonr(feats_np[train_idx, fi], df_mag[train_idx])[0] for fi in range(feat_dim)])
r_te = np.array([pearsonr(feats_np[test_idx,  fi], df_mag[test_idx] )[0] for fi in range(feat_dim)])
r_fu = np.array([pearsonr(feats_np[:, fi], df_mag)[0] for fi in range(feat_dim)])

fig, axes = plt.subplots(1, 2, figsize=(max(12, feat_dim*0.7), 4))
x = np.arange(feat_dim)
ax = axes[0]
ax.bar(x - 0.25, r_tr, 0.25, color="tomato",    alpha=0.8, label="train")
ax.bar(x,        r_fu, 0.25, color="gray",       alpha=0.6, label="full")
ax.bar(x + 0.25, r_te, 0.25, color="steelblue",  alpha=0.8, label="test")
ax.set_xticks(x); ax.set_xticklabels(feat_names, rotation=45, ha="right")
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("Pearson R with |ΔF|"); ax.set_title("Feature → |ΔF| correlation")
ax.legend()

# Feature agreement score: how similar are train and test correlations?
r_agreement = pearsonr(r_tr, r_te)[0]
ax2 = axes[1]
ax2.scatter(r_tr, r_te, c=range(feat_dim), cmap="tab20", s=60, zorder=3)
for i, n in enumerate(feat_names):
    ax2.annotate(n, (r_tr[i], r_te[i]), fontsize=7, ha="left")
lim = max(abs(r_tr).max(), abs(r_te).max()) * 1.1
ax2.plot([-lim, lim], [-lim, lim], "k--", lw=1)
ax2.set_xlabel("train feature R"); ax2.set_ylabel("test feature R")
ax2.set_title(f"Feature agreement train↔test  R={r_agreement:.4f}")
ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\nFeature agreement score: {r_agreement:.4f}")
print("  > 0.9 → same structure in both regions → MLP can generalise")
print("  < 0.7 → features behave differently   → harder to generalise")

## Section 5 — MLP training in-notebook

Trains the Lagrangian MLP from scratch here. No external script needed.  
Watch train R vs test R converge — the gap tells you if it overfits the sub-region.

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
lag_model = make_lagrangian_corrector(
    hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS, output_dim=3
)

vel_feat = vel if USE_VEL else jnp.zeros_like(vel)
weights  = compute_sample_weights(
    pos_lr, det_D_np, MESH_LR, sc_boost=1.0, density_boost=0.0, density_gamma=0.5
)

schedule  = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=LR, warmup_steps=100, decay_steps=N_STEPS
)
optimizer = optax.adamw(schedule, weight_decay=1e-4)

rng       = jax.random.PRNGKey(0)
params    = lag_model.init(rng, feats_all[train_idx[:4]], vel_feat[train_idx[:4]],
                            jnp.array(a))
opt_state = optimizer.init(params)

train_step = make_train_step(lag_model, optimizer)
n_params   = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"MLP: {n_params:,} params  hidden={HIDDEN_DIM}×{N_LAYERS}  feat_dim={feats_np.shape[1]}")

# ── Helper: R for a region ─────────────────────────────────────────────────────
def region_r(params, idx):
    pred = np.asarray(jax.device_get(
        lag_model.apply(params, feats_all[idx], vel_feat[idx], jnp.array(a))
    ))
    return float(np.mean([pearsonr(pred[:,c], delta_f_np[idx,c])[0] for c in range(3)]))

# ── Training targets ──────────────────────────────────────────────────────────
# Target for MLP: full ΔF (single-stage), or residual ΔF-CNN if two-stage
CNN_MODEL, CNN_PARAMS = None, None
cnn_pred_np = np.zeros_like(delta_f_np)

if CNN_CKPT_PATH:
    CNN_MODEL, CNN_PARAMS = _load_cnn_massres_checkpoint(CNN_CKPT_PATH)
    _cnn_jit = jax.jit(lambda pos, vel, a:
        compute_cnn_massres_correction(CNN_MODEL, CNN_PARAMS, pos, vel, a, MESH_LR))
    vel_lr = vel * SCALE_TO_LR
    cnn_pred_np = np.asarray(jax.device_get(
        _cnn_jit(pos_lr, vel_lr, jnp.array(a))
    ))
    cnn_r = float(np.mean([pearsonr(cnn_pred_np[:,c], delta_f_np[:,c])[0] for c in range(3)]))
    print(f"CNN loaded  R_mean={cnn_r:.4f}")
    target_all = delta_f - jnp.array(cnn_pred_np)   # residual for MLP
else:
    target_all = delta_f   # full ΔF

target_tr = target_all[train_idx]
wt_tr     = weights[train_idx]
vel_tr    = vel_feat[train_idx]
feats_tr  = feats_all[train_idx]

print(f"MLP target |residual| mean = {float(jnp.mean(jnp.sqrt(jnp.sum(target_all**2, axis=-1)))):.4e}")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
history = {"step": [], "loss": [], "train_r": [], "test_r": [], "total_train_r": [], "total_test_r": []}
best_test_r, best_params = -1.0, None

t0 = time.time()
for step in range(N_STEPS):
    params, opt_state, loss = train_step(
        params, opt_state, feats_tr, vel_tr, jnp.array(a), target_tr, wt_tr
    )

    if step % LOG_EVERY == 0 or step == N_STEPS - 1:
        mlp_r_tr = region_r(params, train_idx)
        mlp_r_te = region_r(params, test_idx)

        # Total (CNN + MLP) vs full ΔF
        if CNN_MODEL is not None:
            mlp_pred_tr = np.asarray(jax.device_get(
                lag_model.apply(params, feats_all[train_idx], vel_feat[train_idx], jnp.array(a))
            )) + cnn_pred_np[train_idx]
            mlp_pred_te = np.asarray(jax.device_get(
                lag_model.apply(params, feats_all[test_idx],  vel_feat[test_idx],  jnp.array(a))
            )) + cnn_pred_np[test_idx]
            tot_r_tr = float(np.mean([pearsonr(mlp_pred_tr[:,c], delta_f_np[train_idx,c])[0] for c in range(3)]))
            tot_r_te = float(np.mean([pearsonr(mlp_pred_te[:,c], delta_f_np[test_idx,c] )[0] for c in range(3)]))
        else:
            tot_r_tr, tot_r_te = mlp_r_tr, mlp_r_te

        history["step"].append(step)
        history["loss"].append(float(loss))
        history["train_r"].append(mlp_r_tr)
        history["test_r"].append(mlp_r_te)
        history["total_train_r"].append(tot_r_tr)
        history["total_test_r"].append(tot_r_te)

        if tot_r_te > best_test_r:
            best_test_r = tot_r_te
            best_params = hk.data_structures.to_mutable_dict(params)

        if step % (LOG_EVERY * 4) == 0 or step == N_STEPS - 1:
            elapsed = time.time() - t0
            print(f"step {step:4d} | loss={float(loss):.3e} | "
                  f"MLP R  train={mlp_r_tr:.3f} test={mlp_r_te:.3f} gap={mlp_r_tr-mlp_r_te:+.3f} | "
                  f"total R  train={tot_r_tr:.3f} test={tot_r_te:.3f} | "
                  f"{elapsed:.0f}s")

params = hk.data_structures.to_immutable_dict(best_params)
print(f"\nBest total test R = {best_test_r:.4f}  (step with best checkpoint)")

In [ ]:
# ── Learning curve ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(history["step"], history["train_r"], "tomato",    lw=2, label="MLP train R (vs residual)")
ax.plot(history["step"], history["test_r"],  "lightcoral", lw=2, ls="--", label="MLP test R")
if CNN_MODEL is not None:
    ax.plot(history["step"], history["total_train_r"], "darkblue", lw=2, label="Total train R (CNN+MLP)")
    ax.plot(history["step"], history["total_test_r"],  "steelblue",lw=2, ls="--", label="Total test R")
    ax.axhline(cnn_r, color="green", ls=":", lw=1.5, label=f"CNN alone R={cnn_r:.4f}")
ax.set_xlabel("step"); ax.set_ylabel("Pearson R")
ax.set_title("Learning curve"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
gap = np.array(history["total_train_r"]) - np.array(history["total_test_r"])
ax.plot(history["step"], gap, "purple", lw=2)
ax.axhline(0,    color="k",   ls="--", lw=0.8)
ax.axhline(0.05, color="orange", ls=":", lw=1, label="0.05 threshold (GOOD)")
ax.axhline(0.15, color="red",  ls=":", lw=1, label="0.15 threshold (LARGE GAP)")
ax.set_xlabel("step"); ax.set_ylabel("train R − test R")
ax.set_title("Generalisation gap over training"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

final_tr = history["total_train_r"][-1]
final_te = history["total_test_r"][-1]
print(f"Final:  train R={final_tr:.4f}  test R={final_te:.4f}  gap={final_tr-final_te:+.4f}")

## Section 6 — What did the MLP learn?

Scatter plots + spatial error maps to diagnose *where* the MLP helps.

In [ ]:
# ── Final predictions ─────────────────────────────────────────────────────────
mlp_pred_all = np.asarray(jax.device_get(
    jax.jit(lag_model.apply)(params, feats_all, vel_feat, jnp.array(a))
))
total_pred_all = cnn_pred_np + mlp_pred_all   # zero-safe if no CNN

# ── Scatter: ΔF target vs predicted ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, (idx, label, cmap_name) in enumerate([
    (train_idx, "Train", "Blues"),
    (test_idx,  "Test",  "Greens"),
]):
    ss = np.random.default_rng(row).choice(len(idx), min(30_000, len(idx)), replace=False)
    idx_ss = idx[ss]
    for ci, comp in enumerate(["x", "y", "z"]):
        tgt  = delta_f_np[idx_ss, ci]
        pred = total_pred_all[idx_ss, ci]
        lim  = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15
        ax   = axes[row, ci]
        h    = ax.hexbin(tgt, pred, gridsize=55, cmap=cmap_name,
                         norm=LogNorm(), mincnt=1, extent=[-lim,lim,-lim,lim])
        plt.colorbar(h, ax=ax)
        ax.plot([-lim,lim],[-lim,lim],"r--",lw=1)
        r_here = pearsonr(tgt, pred)[0]
        ax.set_title(f"{label} F_{comp}  R={r_here:.4f}")
        ax.set_xlabel("ΔF target"); ax.set_ylabel("ΔF predicted")
plt.suptitle(f"Total prediction scatter (CNN+MLP)  a={a:.3f}", fontsize=11)
plt.tight_layout(); plt.show()

# ── Residual error: where does prediction fail? ───────────────────────────────
err_all  = np.sqrt(np.sum((total_pred_all - delta_f_np)**2, axis=-1))
err_base = np.sqrt(np.sum((np.zeros_like(delta_f_np) - delta_f_np)**2, axis=-1))  # |ΔF| itself
improv   = 1 - err_all / (err_base + 1e-12)   # per-particle improvement fraction

slab_mask = np.abs(pos_np[:, 2] % N_PART - N_PART//2) < max(1, N_PART//20)
px, py    = pos_np[slab_mask, 0], pos_np[slab_mask, 1]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (vals, label, cmap) in zip(axes, [
    (err_all[slab_mask],   "prediction error",       "Reds"),
    (improv[slab_mask],    "per-particle improvement","RdYlGn"),
    ((det_D_np<0).astype(float)[slab_mask], "SC fraction", "Purples"),
]):
    hb = ax.hexbin(px, py, C=vals, gridsize=55, cmap=cmap, reduce_C_function=np.mean)
    plt.colorbar(hb, ax=ax); ax.set_title(label)
plt.suptitle(f"Spatial error maps  a={a:.3f}", fontsize=11)
plt.tight_layout(); plt.show()

print(f"Per-particle improvement:  mean={improv.mean():.3f}  "
      f"frac_improved={np.mean(improv>0):.2%}")

## Section 7 — Context window scan

Train the MLP with different `n_shell` values to find the receptive field that maximises test R.  
A plateau in test R as n_shell grows means adding more neighbors stops helping — the signal is local.

In [ ]:
SHELLS_TO_TEST = [0, 1, 2]   # 0=axis(6), 1=+18, 2=+56 neighbours
N_STEPS_SCAN   = 1000         # fewer steps for quick scan

shell_results = []
for n_sh in SHELLS_TO_TEST:
    # Build features with this shell size
    if n_sh > 0:
        ext_i, ext_o, sh_sl = get_shell_neighbor_indices(N_PART, n_sh)
    else:
        ext_i, ext_o, sh_sl = None, None, ()
    feats_sh, _ = snapshot_features(
        pos, neighbor_idx, N_PART, USE_STRAIN, USE_INVS,
        ext_i, ext_o, "mean_var", sh_sl,
    )
    fd = int(feats_sh.shape[1])
    print(f"n_shell={n_sh}  feat_dim={fd}")

    # Quick train
    model_sh = make_lagrangian_corrector(hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS, output_dim=3)
    sched_sh = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=LR, warmup_steps=50, decay_steps=N_STEPS_SCAN
    )
    opt_sh   = optax.adamw(sched_sh, weight_decay=1e-4)
    p_sh     = model_sh.init(jax.random.PRNGKey(0),
                              feats_sh[train_idx[:4]], vel_feat[train_idx[:4]], jnp.array(a))
    os_sh    = opt_sh.init(p_sh)
    ts_sh    = make_train_step(model_sh, opt_sh)

    vf = vel_feat
    for step in range(N_STEPS_SCAN):
        p_sh, os_sh, _ = ts_sh(
            p_sh, os_sh,
            feats_sh[train_idx], vf[train_idx], jnp.array(a),
            target_all[train_idx], weights[train_idx]
        )

    def _r(idx):
        pred = np.asarray(jax.device_get(model_sh.apply(p_sh, feats_sh[idx], vf[idx], jnp.array(a))))
        total = cnn_pred_np[idx] + pred
        return float(np.mean([pearsonr(total[:,c], delta_f_np[idx,c])[0] for c in range(3)]))

    r_tr = _r(train_idx); r_te = _r(test_idx)
    n_p  = sum(x.size for x in jax.tree_util.tree_leaves(p_sh))
    shell_results.append({"n_shell": n_sh, "feat_dim": fd, "n_params": n_p,
                           "train_R": r_tr, "test_R": r_te, "gap": r_tr - r_te})
    print(f"  total train R={r_tr:.4f}  test R={r_te:.4f}  gap={r_tr-r_te:+.4f}  ({n_p:,} params)")

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
sh_arr = [r["n_shell"]   for r in shell_results]
tr_arr = [r["train_R"]   for r in shell_results]
te_arr = [r["test_R"]    for r in shell_results]
ax.plot(sh_arr, tr_arr, "o-", c="tomato",    lw=2, label="train R")
ax.plot(sh_arr, te_arr, "o-", c="steelblue", lw=2, label="test R")
if CNN_CKPT_PATH:
    ax.axhline(cnn_r, color="green", ls=":", lw=1.5, label=f"CNN alone R={cnn_r:.4f}")
ax.set_xlabel("n_shell"); ax.set_ylabel("total Pearson R (CNN+MLP)")
ax.set_title("Context window scan: n_shell vs generalisation")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\n" + "─"*65)
print(f"{'n_shell':>8} {'feat_dim':>10} {'n_params':>10} {'train_R':>9} {'test_R':>9} {'gap':>8}")
print("─"*65)
for r in shell_results:
    print(f"{r['n_shell']:>8} {r['feat_dim']:>10} {r['n_params']:>10,} "
          f"{r['train_R']:>9.4f} {r['test_R']:>9.4f} {r['gap']:>8.4f}")

## Section 8 — Summary table

In [ ]:
import pandas as pd

# Collect all test-region results
summary_rows = []

# Baselines
for name, pred in [("predict-zero",  pred_zero),
                   ("predict-mean",  pred_mean),
                   ("linear feats→ΔF", np.vstack([
                       np.zeros((len(train_idx), 3)),   # placeholder train
                       pred_lm_df_te
                   ]))]:
    pass   # already printed above — collect from results dict

# Rebuild cleanly
rows_summary = []
for tag, pred_te in [
    ("predict-zero",          pred_zero[test_idx]),
    ("predict-train-mean",    pred_mean[test_idx]),
    ("linear: feats→ΔF",     pred_lm_df_te),
    ("MLP (this notebook)",   total_pred_all[test_idx]),
]:
    r  = r_mean_3d(pred_te, delta_f_np[test_idx])
    mi = mse_improv(pred_te, f_fine_np[test_idx], f_coarse_np[test_idx])
    rows_summary.append({"method": tag, "test_R": r, "MSE_improv%": mi * 100})

df_sum = pd.DataFrame(rows_summary)
df_sum = df_sum.sort_values("test_R", ascending=False)
print("\n── All methods — test region ──────────────────────────────")
print(df_sum.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nKey takeaways:")
best_r = df_sum["test_R"].max()
print(f"  Best test R = {best_r:.4f}")
if CNN_CKPT_PATH:
    print(f"  CNN alone   = {cnn_r:.4f}")
mlp_row = df_sum[df_sum.method == "MLP (this notebook)"].iloc[0]
print(f"  MLP gain over predict-zero: {mlp_row['test_R'] - 0:.4f} R  /  "
      f"{mlp_row['MSE_improv%']:.1f}% MSE")